In [ ]:
import pandas as pd
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility
from ollama import Client

# ==========================================
# 1. Configurações e Conexões (Rede Docker)
# ==========================================
minio_storage_options = {
    "key": "minioadmin",
    "secret": "minioadmin123",
    "client_kwargs": {"endpoint_url": "http://minio:9000"}
}

# Conecta ao Milvus e Ollama usando o DNS interno dos containers
connections.connect("default", host="milvus", port="19530")
ollama_client = Client(host='http://ollama:11434')

print("✅ Conectado ao Milvus e Ollama!")

# ==========================================
# 2. Preparar a Coleção no Milvus
# ==========================================
collection_name = "triageai_knowledge_base"
dimensao_nomic = 768 # O modelo nomic-embed-text gera vetores de 768 dimensões

# Se a coleção já existir (ex: rodando o script de novo), nós a apagamos para recriar
if utility.has_collection(collection_name):
    utility.drop_collection(collection_name)

# Definindo o Schema do banco vetorial
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="disease", dtype=DataType.VARCHAR, max_length=200),
    FieldSchema(name="texto_rag", dtype=DataType.VARCHAR, max_length=2500),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=dimensao_nomic)
]
schema = CollectionSchema(fields, description="Base de conhecimento de doenças e sintomas para o TriageAI")
collection = Collection(name=collection_name, schema=schema)

print(f"✅ Coleção '{collection_name}' criada no Milvus!")

# ==========================================
# 3. Ler dados da Gold e Gerar Embeddings
# ==========================================
path_gold = "s3://gold/textos_rag.csv"
df_gold = pd.read_csv(path_gold, storage_options=minio_storage_options)
print(f"✅ Dados da Gold carregados! Total: {len(df_gold)} registros.")

# Vamos preparar as listas para inserção
doencas = df_gold['diseases'].tolist()
textos = df_gold['texto'].tolist()
embeddings = []

print("⏳ Gerando embeddings via Ollama (isso pode levar alguns minutos)...")
for texto in textos:
    # Chama o LLM local para transformar o texto em vetor
    response = ollama_client.embeddings(model='nomic-embed-text', prompt=texto)
    embeddings.append(response['embedding'])

# ==========================================
# 4. Inserção e Indexação (O Entregável da Sprint)
# ==========================================
print("⏳ Inserindo dados no Milvus...")
data_to_insert = [
    doencas,
    textos,
    embeddings
]
collection.insert(data_to_insert)

# Cria o índice de busca vetorial (necessário para buscas rápidas)
index_params = {
    "metric_type": "COSINE", # Distância por cosseno é ideal para textos
    "index_type": "HNSW",    # Algoritmo de busca aproximada de alta performance
    "params": {"M": 8, "efConstruction": 64}
}
collection.create_index(field_name="embedding", index_params=index_params)

# Carrega a coleção na memória para deixá-la pronta para buscas
collection.load()

print("🚀 SPRINT 5 CONCLUÍDA: Index funcional criado e carregado com sucesso no Milvus!")

✅ Conectado ao Milvus e Ollama!
✅ Coleção 'triageai_knowledge_base' criada no Milvus!
✅ Dados da Gold carregados! Total: 246945 registros.
⏳ Gerando embeddings via Ollama (isso pode levar alguns minutos)...
